# Combine Extracted Kline CSVs By Time Range

Interactive notebook for combine + integrity checks.
All combine logic lives in `data_processing/combine_extracted_klines_logic.py`.


In [ ]:
from pathlib import Path
import sys

CWD = Path.cwd()
if (CWD / "data_processing").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "data_processing").exists():
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = CWD

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_processing.combine_extracted_klines_logic import CombineConfig, combine_extracted_klines

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
# -----------------------------
# Configuration
# -----------------------------
SYMBOL = "BTCUSDT"
INTERVAL = "1s"
START_YYYY_MM = "2018-01"
END_YYYY_MM = "2020-02"

# Output options: 'csv' or 'parquet'
OUTPUT_FORMAT = "csv"
OUTPUT_NAME = ""
SUMMARY_LOG_NAME = ""

# Column selection: use one strategy
KEEP_COLUMNS = []
DROP_COLUMNS = []

# Optional overrides. Keep as None for defaults under downloads/test1.
EXTRACTED_DIR_OVERRIDE = None
OUTPUT_DIR_OVERRIDE = None

# Parquet + integrity reporting options
PARQUET_COMPRESSION = "snappy"
PARQUET_CHUNK_SIZE = 200000
MAX_EXAMPLES_PER_ISSUE = 15

config = CombineConfig(
    symbol=SYMBOL,
    interval=INTERVAL,
    start_yyyy_mm=START_YYYY_MM,
    end_yyyy_mm=END_YYYY_MM,
    project_root=PROJECT_ROOT,
    extracted_dir=Path(EXTRACTED_DIR_OVERRIDE) if EXTRACTED_DIR_OVERRIDE else None,
    output_dir=Path(OUTPUT_DIR_OVERRIDE) if OUTPUT_DIR_OVERRIDE else None,
    output_format=OUTPUT_FORMAT,
    output_name=OUTPUT_NAME,
    summary_log_name=SUMMARY_LOG_NAME,
    keep_columns=KEEP_COLUMNS,
    drop_columns=DROP_COLUMNS,
    parquet_compression=PARQUET_COMPRESSION,
    parquet_chunk_size=PARQUET_CHUNK_SIZE,
    max_examples_per_issue=MAX_EXAMPLES_PER_ISSUE,
)

result = combine_extracted_klines(config)

print(f"Combined output : {result.combined_output_path}")
print(f"Summary log     : {result.summary_log_path}")
print(f"Files processed : {result.files_processed}")
print(f"Rows read       : {result.rows_read}")
print(f"Rows written    : {result.rows_written}")
print(f"Issue types     : {len(result.issue_counts)}")


In [ ]:
print("Summary log preview (first 40 lines):")
with result.summary_log_path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i >= 39:
            break

if config.output_format.lower().strip() == "csv":
    print("\nCombined CSV preview (first 6 lines):")
    with result.combined_output_path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            print(line.rstrip())
            if i >= 5:
                break
else:
    size_mb = result.combined_output_path.stat().st_size / (1024 * 1024)
    print(f"\nParquet file size: {size_mb:.2f} MB")
